# Parte 1 – Scraping de decretos de emergencia

In [1]:
import time, re
import requests
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

#### Paso 1 – robots.txt de gob.pe

In [3]:
headers = {"User-Agent": "Mozilla/5.0"}

robots = requests.get("https://www.gob.pe/robots.txt", headers=headers, timeout=30)
print(robots.text)

# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



##### Análisis del robots.txt de gob.pe
**¿Qué prohíbe?** Para cualquier bot (`User-agent: *`), prohíbe acceder a la sección `/admin/` y `sheet=`. Además, bloquea por completo el sitio para `AhrefsBot`.

**¿Está permitida la página de búsquedas?** La ruta `/busquedas` no aparece en ninguna regla `Disallow`, por lo que se conluye que esta permitida.

**¿Por qué igual conviene hacer pausas?** Aunque no se nos exige un `Crawl-delay`, conviene hacer pausas para: 1. evitar que nos bloqueen y 2. Saturar el servicio que brinda gob.pe, porque ademas afectaria a los ciudadanos. Por otro lado, la pagina sí le pide a otros bots `Crawl-delay`: 5 segundos a `GPTBot` y 2 segundos a `OAI-SearchBot`.

#### Paso 2 – Abrir con Selenium la búsqueda de un mes

In [4]:
BASE = ("https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm"
        "&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent")

url_mes = BASE + "&desde=01-01-2024&hasta=31-01-2024"

navegador = webdriver.Chrome()
navegador.get(url_mes)

WebDriverWait(navegador, 30).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "article")))
time.sleep(3)  # margen extra: la página sigue dibujando resultados
print("Contenido cargado")
print("Artículos encontrados:", len(navegador.find_elements(By.TAG_NAME, "article")))

Contenido cargado
Artículos encontrados: 11


#### Paso 3 – Leer el total de resultados ("N Resultados")

In [5]:
# el total aparece arriba de la lista como "N Resultados", dentro de <h2 role="status"><span>
contador = WebDriverWait(navegador, 30).until(
    EC.visibility_of_element_located((By.CSS_SELECTOR, "h2[role='status'] span")))
texto_total = contador.text                             # ej. "11 Resultados"
total_resultados = int(re.sub(r"\D", "", texto_total))  # nos quedamos solo con los dígitos

print("Texto en la página:", texto_total)
print("Total de resultados:", total_resultados)

Texto en la página: 11 Resultados
Total de resultados: 11


#### Paso 4 – Extraer los datos de cada resultado

In [6]:
def extraer_articulo(art):
    # enlace principal: trae el número de la norma y su URL
    link = art.find_element(By.CSS_SELECTOR, "a.link-principal")

    # la fecha es la línea que empieza con "Publicado:"
    fecha = ""
    for linea in art.text.split("\n"):
        if linea.startswith("Publicado:"):
            fecha = linea.replace("Publicado:", "").strip()

    # el título es el párrafo debajo de la fecha (viene cortado con "...")
    titulo = art.find_element(By.TAG_NAME, "p").text

    return {
        "numero": link.text.strip(),
        "fecha": fecha,
        "titulo": titulo.strip(),
        "enlace": link.get_attribute("href"),
    }

In [7]:
# prueba con los resultados de enero que ya están abiertos en el navegador
articulos = navegador.find_elements(By.TAG_NAME, "article")
filas_enero = []
for art in articulos:
    filas_enero.append(extraer_articulo(art))

print("Normas extraídas:", len(filas_enero))
pd.DataFrame(filas_enero)

Normas extraídas: 11


,numero,fecha,titulo,enlace
0,Decreto Supremo N.° 010-2024-PCM,30 de enero de 2024,Prorrogar por el termino de sesenta (60) días ...,https://www.gob.pe/institucion/pcm/normas-lega...
1,Decreto Supremo N.° 009-2024-PCM,29 de enero de 2024,Prorrogar por el termino de sesenta (60) días ...,https://www.gob.pe/institucion/pcm/normas-lega...
2,Decreto Supremo N.° 008-2024-PCM,29 de enero de 2024,Declarar por el termino de treinta (30) días c...,https://www.gob.pe/institucion/pcm/normas-lega...
3,Decreto Supremo N.° 007-2024-PCM,29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,Decreto Supremo N.° 006-2024-PCM,29 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
5,Decreto Supremo N.° 005-2024-PCM,29 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
6,Decreto Supremo N.° 004-2024-PCM,26 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
7,Resolución N.° 0214-2024-TCE-S6,18 de enero de 2024,N° 10304103618); por su responsabilidad al hab...,https://www.gob.pe/institucion/oece/normas-leg...
8,Decreto Supremo N.° 003-2024-PCM,10 de enero de 2024,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
9,Decreto Supremo N.° 002-2024-PCM,9 de enero de 2024,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...


#### Paso 5 – Recorrer los meses de la temporada (enero a mayo 2024)

In [8]:
def scrapear_mes(navegador, desde, hasta):
    navegador.get(BASE + f"&desde={desde}&hasta={hasta}")

    # esperar al total o, si no hay resultados, al aviso "No se encontraron resultados"
    WebDriverWait(navegador, 30).until(EC.any_of(
        EC.visibility_of_element_located((By.CSS_SELECTOR, "h2[role='status'] span")),
        EC.visibility_of_element_located((By.XPATH, "//h2[contains(text(), 'No se encontraron resultados')]"))
    ))
    time.sleep(3)  # margen extra: la página sigue dibujando resultados

    # sin resultados la página no muestra el contador
    contador = navegador.find_elements(By.CSS_SELECTOR, "h2[role='status'] span")
    if len(contador) == 0:
        return 0, []
    total = int(re.sub(r"\D", "", contador[0].text))

    filas = []
    for art in navegador.find_elements(By.TAG_NAME, "article"):
        filas.append(extraer_articulo(art))
    return total, filas

In [9]:
# prueba: el 1 de enero de 2024 no tiene resultados
print(scrapear_mes(navegador, "01-01-2024", "01-01-2024"))
time.sleep(2)

(0, [])


In [10]:
# meses de la temporada de lluvias (2024 es bisiesto: febrero termina el 29)
meses = [
    ("enero",   "01-01-2024", "31-01-2024"),
    ("febrero", "01-02-2024", "29-02-2024"),
    ("marzo",   "01-03-2024", "31-03-2024"),
    ("abril",   "01-04-2024", "30-04-2024"),
    ("mayo",    "01-05-2024", "31-05-2024"),
]

todas = []
control = []

for mes, desde, hasta in meses:
    total, filas = scrapear_mes(navegador, desde, hasta)
    for fila in filas:
        todas.append(fila)
    control.append({"mes": mes,
                    "resultados_totales": total,
                    "resultados_extraidos": len(filas)})
    print(f"{mes}: {total} resultados, {len(filas)} extraídos")
    time.sleep(2)  # pausa entre meses para no saturar gob.pe

navegador.quit()
print("Navegador cerrado")
print("Normas extraídas en la temporada:", len(todas))

enero: 11 resultados, 11 extraídos
febrero: 11 resultados, 11 extraídos
marzo: 11 resultados, 11 extraídos
abril: 15 resultados, 15 extraídos
mayo: 3 resultados, 3 extraídos
Navegador cerrado
Normas extraídas en la temporada: 51


#### Paso 6 – Verificación

In [11]:
# control por mes: el total de la página debe coincidir con lo extraído
df_control = pd.DataFrame(control)
df_control["coincide"] = df_control["resultados_totales"] == df_control["resultados_extraidos"]
print("¿Coinciden todos los meses?:", df_control["coincide"].all())
df_control

¿Coinciden todos los meses?: True


,mes,resultados_totales,resultados_extraidos,coincide
0,enero,11,11,True
1,febrero,11,11,True
2,marzo,11,11,True
3,abril,15,15,True
4,mayo,3,3,True


#### Paso 7 – Eliminar normas repetidas

In [ ]:
normas = pd.DataFrame(todas) #convierte la lista de resultados en una tabla
antes = len(normas) # se guardan la cantidad de filas antes de limpiar para hacer la comparación
normas = normas.drop_duplicates(subset="enlace").reset_index(drop=True) #drop_duplicates: elimina las filas cuyo enlace ya apareció antes y conserva la primera. 
#con reset_index: renumera las filas de 0 en adelante, para que no queden huecos en el índice.
print(f"Filas antes: {antes} | después: {len(normas)} | repetidas eliminadas: {antes - len(normas)}")  
# muestra el resultado: 51 antes, 51 después, 0 repetidas.


Filas antes: 51 | después: 51 | repetidas eliminadas: 0
